# Training an AI to detect seizures in EEG

This notebook trains the models from the **eeg-seizure-detection** project on Google's computers.

**How to use it**
1. Run the cells from top to bottom: click a cell and press **Shift + Enter**, or use **Runtime → Run all**.
2. The first code cell downloads the project from GitHub; no upload is needed.
3. No GPU is needed. The normal free runtime is fine.

**What happens**
| Part | What it does | Time |
|---|---|---|
| 1–2 | Get the code and the EEG data | about 1 min |
| 3 | Look at the brain waves | seconds |
| 4 | Train a model yourself, step by step | about 1 min |
| 5 | Run every experiment from the project | about 10 min |
| 6–7 | See the charts and save the results | about 1 min |

Colab deletes files when the session ends, so download your results in Part 7.

## Part 1: Get the project code

This downloads the project from GitHub automatically.
To use a local copy instead, set `REPO_URL = ""` and upload **eeg-seizure-detection.zip** when the button appears.

In [ ]:
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/sanjambahl/eeg-seizure-detection"   # set to "" to upload the project zip instead
PROJECT = "eeg-seizure-detection"

if Path.cwd().name != PROJECT:
    if not os.path.isdir(PROJECT):
        if REPO_URL:
            subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)
        else:
            from google.colab import files
            print("Choose eeg-seizure-detection.zip ...")
            uploaded = files.upload()
            zip_name = next(name for name in uploaded if name.endswith(".zip"))
            zipfile.ZipFile(zip_name).extractall(".")
    os.chdir(PROJECT)

sys.path.insert(0, os.getcwd())
print("Working in:", os.getcwd())
print("Files:", sorted(os.listdir()))

## Part 2: Get the EEG data

This downloads the **Bonn EEG dataset** (Andrzejak et al., 2001) from its official page at Universitat Pompeu Fabra.
If that page doesn't respond, it uses a copy on GitHub instead.

The data are free for research and education. Cite: Andrzejak RG, Lehnertz K, Rieke C, Mormann F, David P, Elger CE (2001), *Physical Review E* 64, 061907.

In [ ]:
DATA = "data/bonn"

def count_recordings():
    return sum(1 for p in Path(DATA).rglob("*") if p.suffix.lower() == ".txt") if os.path.isdir(DATA) else 0

if count_recordings() < 500:
    result = subprocess.run([sys.executable, "scripts/download_data.py"], capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode != 0 or count_recordings() < 500:
        print("Official download didn't work, so using the GitHub copy instead...")
        shutil.rmtree(DATA, ignore_errors=True)
        if not os.path.isdir("/tmp/bonn_mirror"):
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/RYH2077/EEG-Epilepsy-Datasets", "/tmp/bonn_mirror"], check=True)
        zipfile.ZipFile("/tmp/bonn_mirror/Bonn EEG dataset.zip").extractall(DATA)

print("Recordings found:", count_recordings(), "(should be 500)")

## Part 3: Look at the brain waves

There are 500 recordings, each 23.6 seconds long, in five groups of 100:

| Group | Who / where | Recorded from |
|---|---|---|
| A | Healthy, eyes open | Scalp |
| B | Healthy, eyes closed | Scalp |
| C | Epilepsy, between seizures, side opposite the seizure zone | Inside the skull |
| D | Epilepsy, between seizures, inside the seizure zone | Inside the skull |
| E | Epilepsy, **during a seizure** | Inside the skull |

First, a filter removes slow drift and high-frequency noise, keeping 0.5–40 Hz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eeg.data import load_bonn, bandpass, make_windows, FS, SETS, SET_ORDER

signals, set_labels, recording_ids = load_bonn(DATA)
filtered = bandpass(signals)
print("Recordings:", signals.shape[0], "| samples per recording:", signals.shape[1], "| sampling rate:", FS, "Hz")

t = np.arange(int(8 * FS)) / FS
fig, axes = plt.subplots(5, 1, figsize=(11, 8), sharex=True, sharey=True)
for ax, letter in zip(axes, SET_ORDER):
    first = np.where(set_labels == letter)[0][0]
    ax.plot(t, filtered[first, :len(t)], lw=0.8)
    ax.set_title(f"{SETS[letter][0]} · {SETS[letter][1]}", loc="left", fontsize=10)
axes[-1].set_xlabel("Time (seconds)")
axes[2].set_ylabel("Voltage (µV)")
plt.tight_layout()
plt.show()

## Part 4: Train a model yourself, step by step

### 4a. Turn brain waves into numbers
Each recording is cut into eleven 2-second **windows**. Each window is then described by **18 numbers (features)**, such as how big the waves are, which rhythms dominate, and how regular the pattern is.
The result is a table: one row per window, one column per feature, plus the answer (which group it came from).

In [ ]:
from eeg.features import extract_features, FEATURE_NAMES

windows, window_groups, window_recordings = make_windows(filtered, set_labels, recording_ids, window_sec=2.0)
X = extract_features(windows)

table = pd.DataFrame(X, columns=FEATURE_NAMES)
table["group"] = [SETS[s][0] for s in window_groups]
table["recording"] = window_recordings
print(table.shape[0], "windows x", len(FEATURE_NAMES), "features")
table.head()

### 4b. Pick a question for the AI to answer
Change `TASK` to try a different question:

| `TASK` | Question |
|---|---|
| `"seizure_vs_healthy"` | Seizure (E) vs healthy (A), the classic easy benchmark |
| `"seizure_vs_all"` | Seizure (E) vs every non-seizure group (A–D) |
| `"seizure_vs_interictal"` | Seizure (E) vs the same patients between seizures (C, D) |
| `"healthy_vs_epilepsy"` | Healthy (A, B) vs epilepsy between seizures (C, D) |
| `"zone_vs_opposite"` | Seizure zone (D) vs opposite side (C), the hard one |
| `"five_class"` | Sort all five groups |

In [ ]:
from eeg.experiments import TASKS, task_data

TASK = "seizure_vs_all"     # <- try changing this

X_task, y, recs, _ = task_data(TASK, X, window_groups, window_recordings)
print("Question:", TASKS[TASK][2])
print("Windows used:", len(y), "| windows per answer:", np.bincount(y).tolist())

### 4c. Train one model and test it on recordings it has never seen
We hold back 20% of the **recordings** for testing. Every window from a held-back recording stays out of training.
`model.fit(...)` is the training step: the model looks at the training rows and their answers, and adjusts itself until it makes as few mistakes as it can.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, test_idx = next(splitter.split(X_task, y, recs))
assert not set(recs[train_idx]) & set(recs[test_idx]), "a recording leaked into both sets"
print("Training on", len(set(recs[train_idx])), "recordings; testing on", len(set(recs[test_idx])), "unseen recordings")

model = HistGradientBoostingClassifier(class_weight="balanced", random_state=0)
model.fit(X_task[train_idx], y[train_idx])            # <- this is the training

predictions = model.predict(X_task[test_idx])
print("Balanced accuracy on unseen recordings: %.1f%%" % (100 * balanced_accuracy_score(y[test_idx], predictions)))
print("Confusion matrix (rows = true answer, columns = model's guess):")
print(confusion_matrix(y[test_idx], predictions))

### 4d. Compare all five models fairly (cross-validation)
One split can be lucky or unlucky. So the recordings are split into 5 piles: train a fresh model on 4, test on the 5th, and rotate. Then reshuffle and repeat 3 times, for 15 tests per model.

In [ ]:
from eeg.experiments import make_models, cross_validate

rows = []
for name, m in make_models().items():
    r = cross_validate(m, X_task, y, recs, split="grouped", n_splits=5, seeds=(0, 1, 2))
    rows.append({"model": name,
                 "accuracy per window (%)": 100 * np.mean(r["balanced_accuracy"]),
                 "accuracy per recording (%)": 100 * np.mean(r["recording_balanced_accuracy"])})
    print("done:", name)
pd.DataFrame(rows).round(1)

### 4e. See data leakage happen
If windows are shuffled one by one, pieces of the same recording land in both training and testing. The test then gets easier than it should be.

In [ ]:
gb = make_models()["Gradient boosting"]
honest = np.mean(cross_validate(gb, X_task, y, recs, split="grouped")["balanced_accuracy"]) * 100
leaky = np.mean(cross_validate(gb, X_task, y, recs, split="leaky")["balanced_accuracy"]) * 100
print(f"Honest (recordings held out): {honest:.1f}%")
print(f"Leaky (windows shuffled):     {leaky:.1f}%")
print(f"Fake boost from leakage:      +{leaky - honest:.1f} points")

## Part 5: Run every experiment from the project
This trains about 1,500 models (every task, every model, honest and leaky testing, plus the extra tests) and takes **about 10 minutes**, then runs the extra analyses (ROC curve, learning curve) in about a minute.
Progress lines appear as it goes.

In [ ]:
def run(cmd):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("Command failed: " + " ".join(cmd))

run([sys.executable, "-u", "run_experiments.py", "--data-dir", DATA])
run([sys.executable, "-u", "extra_analyses.py"])
run([sys.executable, "-u", "make_figures.py"])

## Part 6: Look at the results

In [ ]:
from IPython.display import Image, display

cv = pd.read_csv("results/cv_results.csv")
honest_results = cv[cv.split == "grouped"].pivot(index="task", columns="model", values="balanced_accuracy_mean")
display((100 * honest_results).round(1))

for fig in sorted(Path("figures").glob("*.png")):
    print(fig.name)
    display(Image(filename=str(fig), width=900))

## Part 7: Save your results
This downloads a zip with every results table and chart. Colab deletes files when the session ends.

In [ ]:
zip_path = "/content/eeg_results.zip" if os.path.isdir("/content") else os.path.abspath("../eeg_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("results", "figures"):
        for f in sorted(Path(folder).rglob("*")):
            z.write(f)
print("Saved", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; the zip is at", zip_path)

## Ideas to try next
- Change `TASK` in Part 4b and re-run 4b–4e.
- Change `window_sec=2.0` in Part 4a to `1.0` or `4.0`. Do shorter or longer windows work better?
- Add a sixth model in Part 4d, for example `from sklearn.neighbors import KNeighborsClassifier`.
- Remove the amplitude features (`X[:, 4:]`) and see which tasks still work. That's the "signal size shortcut" finding.